## Set-up

In [ ]:
# Set-up
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

project_day = pd.read_csv('../datasets/project_day.csv')
project_day['date'] = pd.to_datetime(project_day['date'])
display(project_day.head())
display(project_day.info())
feature_col = [
    'total_steps', 'cv_steps', 'active_hours', 
    'early_morning_steps', 'morning_steps',
    'afternoon_steps', 'evening_steps'
]


## Patient Stability Analysis

In [ ]:
# Standard supervised method separation
X = project_day[feature_col].values
y = project_day['label'].values
groups = project_day['id'].values

# Iterate every day through first week, then on a weekly basis
K_days = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28]
day_results = []

gkf = GroupKFold(n_splits = 5) # Timeseries, need to be grouped so no i indiviual in train/test

for K in K_days:
    fold_auc = []
    fold_meta = []

    for fold_i, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups = groups), start = 1):
        train_data = project_day.iloc[train_idx]
        test_data = project_day.iloc[test_idx]
        
        scaler = RobustScaler() # Right skew
        X_train_scaled = scaler.fit_transform(train_data[feature_col])
        y_train = train_data['label'].values
        
        # Inverse-weighting balance 
        healthy = np.sum(y_train == 0)
        ms = np.sum(y_train == 1)
        total = healthy + ms
        class_weight = {
            0: total / (2 * healthy),
            1: total / (2 * ms)
            }
        
        # Logistic Model w/ Inverse Weighting
        log_mod = LogisticRegression(
            class_weight = class_weight,
            max_iter = 1000,
            random_state = 223
            )
        log_mod.fit(X_train_scaled, y_train)

        patient_scores, patient_labels = [], []
        for patient_id, g in test_data.sort_values('date').groupby('id'):
            if len(g) >= K: # Only include patients with enough days to evaluate. Should not be an issue, but fail-safe
                gK = g.head(K)
                X_scaled = scaler.transform(gK[feature_col])
                patient_scores.append(log_mod.predict_proba(X_scaled)[:, 1].mean()) # Avg day-level P(MS) into one score per patient
                patient_labels.append(int(gK['label'].iloc[0]))
        fold_auc.append(roc_auc_score(patient_labels, patient_scores))
    day_results.append({
        'K_days' : K,
        'Mean_AUC' : np.mean(fold_auc),
        'Std_AUC' : np.std(fold_auc)
    })
compiled_results = pd.DataFrame(day_results)

# Begin work on the patient stability plot
plt.figure(figsize = (16, 12))
plt.plot(compiled_results['K_days'],compiled_results['Mean_AUC'], marker = 'o')
# Get them cool CI bands 
plt.fill_between(
    compiled_results['K_days'],
    compiled_results['Mean_AUC'] - compiled_results['Std_AUC'],
    compiled_results['Mean_AUC'] + compiled_results['Std_AUC'],
    alpha = 0.25
)
plt.xlabel('Days Per Patient')
plt.ylabel('Patient-Level ROC-AUC')
plt.ylim(0.5, 1)
plt.tight_layout()
plt.grid(True, alpha = 0.25)
plt.savefig('../output/patient_stability.png') # Save dat figure
plt.show()

# Check Results in CSV form 
display(compiled_results)
compiled_results.to_csv('../output/patient_stability.csv', index = False)